In [ ]:
import pandas as pd
import numpy as np
import hvplot.pandas
from rembox_integration_tools import REMboxDataQuery
from rembox_integration_tools.rembox_analysis import StudyColumn, SeriesColumn

hvplot.extension("bokeh")

CLIENT_ID_ENV_VAR = "REMBOX_INT_CLIENT_ID"
CLIENT_PWD_ENV_VAR = "REMBOX_INT_CLIENT_PWD"
TOKEN_URI = "https://autoqa.vll.se/dpqaauth/connect/token"
API_URI = "https://rembox.vll.se/api"
ORIGIN_URI = "https://rembox.vll.se"

rembox = REMboxDataQuery(
    client_id_environment_variable=CLIENT_ID_ENV_VAR,
    client_secret_environment_variable=CLIENT_PWD_ENV_VAR,
    token_uri=TOKEN_URI,
    api_uri=API_URI,
    origin_uri=ORIGIN_URI
)

valid_study_columns = StudyColumn()
valid_series_columns = SeriesColumn()

In [ ]:
def get_data_from_REMbox(rembox: REMboxDataQuery) -> tuple[pd.DataFrame, pd.DataFrame]:
    valid_study_columns = StudyColumn()
    valid_series_columns = SeriesColumn()
    
    # Rax at NUS
    rembox.filter_options.set_inclusive_tags(
        machines=["U204", "U207", "U208"]
    )

   # exclude Acquisition protocols - positionsgenmomlysning
    rembox.filter_options.set_exclusive_tags(
        acquisition_protocols=['CP_Positioning', 'CP_Barn buk', 'Småskelett']
    )
    
    # From 0-15 years
    rembox.filter_options.patient_age_interval_end_value = 15
    rembox.filter_options.patient_age_interval_end_unit = 'Y'

    # about four month data
    rembox.filter_options.study_time_interval_start_date = "2022-10-01T00:00:00Z"
    rembox.filter_options.study_time_interval_end_date = "2023-01-18T00:00:00Z"



    
    rembox.add_columns(
        columns=[
            valid_study_columns.StudyDateTime,
            valid_study_columns.StudyInstanceUID,
            valid_study_columns.StudyId,
            valid_study_columns.Machine,
            valid_study_columns.AccessionNumber,
            valid_study_columns.StudyDescription,
            valid_study_columns.PatientAge,
            valid_study_columns.DoseAreaProductTotal,
            valid_study_columns.FluoroDoseAreaProductTotal,
            valid_study_columns.AcquisitionDoseAreaProductTotal,
            valid_study_columns.DoseRPTotal,
            valid_study_columns.FluoroDoseRPTotal,
            valid_study_columns.AcquisitionDoseRPTotal,
            valid_study_columns.TotalAcquisitionTime,
            valid_study_columns.TotalFluoroTime,
            valid_study_columns.TotalNumberOfIrradiationEvents,
            valid_study_columns.TotalNumberOfRadiographicFrames,
            valid_study_columns.PerformingPhysicianName,
            valid_study_columns.PerformingPhysicianIdentificationSequence,
            valid_study_columns.PatientDbId,
            valid_series_columns.AcquisitionProtocol,
            valid_series_columns.DoseRP,
            valid_series_columns.DateTimeStarted,
            valid_series_columns.DoseAreaProduct
        ]
    )

    return rembox.run_query()

In [ ]:
study_data, series_data = get_data_from_REMbox(rembox=rembox)

In [ ]:
study = study_data.copy()
series = series_data.copy()
series = series.merge(study, on=['studyInstanceUID'], how="left")

In [ ]:
series['type'] = 'Vuxen'
series.loc[series.acquisitionProtocol.str.contains('0-20kg|20-50kg|0-20 kg|20-40kg|7-10 kg', regex=True),'type'] = 'Barn'

series['month'] = pd.to_datetime(series.dateTimeStarted).dt.strftime('%Y%m')


In [ ]:
series.acquisitionProtocol[series.type == 'Vuxen']

In [ ]:
series.acquisitionProtocol[series.type == 'Barn']

In [ ]:
per_month = series.groupby(['month','type']).count
per_month.hvplot.bar(
    y='doseRP',
    by='type'
    )


In [ ]:

results_study_description = series.pivot_table(
    index=[valid_study_columns.StudyDescription],
    columns='type',
    aggfunc={'type': np.count_nonzero}
)
results_study_description

In [ ]:
per_machine = series.groupby(['machine','type']).count()

per_machine.hvplot.bar(
    y='doseRP',
    by='type'
    )


In [ ]:
per_age = series.groupby(['patientAge','type']).count()

per_age.hvplot.bar(
    y='doseRP',
    by='type'
    )